In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import ConvNeXtXLarge
from tensorflow.keras.applications.convnext import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import joblib
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import time
import os

# Globals
last_image_path = None
last_prediction = None
confidence_score = None

def preprocess_image(image_path):
    """Preprocess image for ConvNeXtXLarge"""
    try:
        # Load and preprocess image
        img = load_img(image_path, target_size=(224, 224))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        img_array = preprocess_input(img_array)
        return img_array
    except Exception as e:
        raise ValueError(f"Error preprocessing image: {str(e)}")

def extract_features(image, feature_extractor):
    """Extract features using ConvNeXtXLarge"""
    features = feature_extractor.predict(image, verbose=0)
    features = features.reshape(features.shape[0], -1)  # Flatten the features
    return features

def predict_class(image_path, feature_extractor, lr_model):
    """Predict bone abnormality with confidence score"""
    global confidence_score
    
    # Preprocess image
    img = preprocess_image(image_path)
    
    # Extract features
    features = extract_features(img, feature_extractor)
    
    # Get prediction probabilities
    probabilities = lr_model.predict_proba(features)[0]
    prediction = lr_model.predict(features)[0]
    confidence = max(probabilities) * 100  # Confidence percentage
    
    confidence_score = confidence
    
    # Convert numeric prediction to class name
    if prediction == 0:
        return "NORMAL", confidence
    else:
        return "ABNORMAL", confidence

def show_image(image_path):
    """Display image in GUI"""
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not load image")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Calculate aspect ratio for better display
    h, w = img_rgb.shape[:2]
    display_size = 350
    if h > w:
        new_h = display_size
        new_w = int(w * display_size / h)
    else:
        new_w = display_size
        new_h = int(h * display_size / w)
    
    img_resized = cv2.resize(img_rgb, (new_w, new_h))
    
    # Clear previous canvas
    for widget in canvas_frame.winfo_children():
        widget.destroy()
    
    # Create new figure with better quality
    fig = plt.Figure(figsize=(5, 5), dpi=100, facecolor='white')
    ax = fig.add_subplot(111)
    ax.imshow(img_resized)
    ax.axis('off')
    ax.set_title("Bone Marrow Image", fontsize=12, pad=10, fontweight='bold')
    
    canvas = FigureCanvasTkAgg(fig, master=canvas_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(expand=True, fill=tk.BOTH)

def browse_and_predict():
    """Browse image and predict"""
    global last_image_path, last_prediction, confidence_score
    
    image_path = filedialog.askopenfilename(
        title="Select Bone Marrow Image",
        filetypes=[("Image Files", "*.jpg *.png *.jpeg *.bmp *.tiff")]
    )
    
    if image_path:
        try:
            # Display image
            show_image(image_path)
            status_label.config(text="Processing image...", fg="blue")
            root.update()
            
            # Start timing
            start_time = time.time()
            
            # Make prediction
            prediction, confidence = predict_class(image_path, feature_extractor, lr_model)
            
            # Calculate processing time
            processing_time = time.time() - start_time
            
            last_image_path = image_path
            last_prediction = prediction
            
            # Show prediction result
            if prediction == "NORMAL":
                result_text = f"✅ PREDICTION: {prediction}"
                result_color = "#27ae60"  # Green
                result_detail = "• The bone appears normal\n• No abnormalities detected\n• Regular checkups recommended"
                recommendation = "Continue regular monitoring"
            else:
                result_text = f"⚠️ PREDICTION: {prediction}"
                result_color = "#e74c3c"  # Red
                result_detail = "• Abnormality detected\n• Please consult a specialist\n• Further examination recommended"
                recommendation = "Consult a medical specialist immediately"
            
            # Update result labels
            result_label.config(text=result_text, fg=result_color)
            detail_label.config(text=result_detail, fg=result_color)
            
            # Add confidence and time info
            info_text = f"📊 Confidence: {confidence:.2f}% | ⏱️ Time: {processing_time:.2f}s"
            info_label.config(text=info_text, fg="#2980b9")
            
            status_label.config(text=f"Prediction completed successfully", fg="green")
            
            # Show detailed results in messagebox
            messagebox.showinfo(
                "Prediction Result",
                f"🦴 Bone Classification Result 🦴\n\n"
                f"Prediction: {prediction}\n"
                f"Confidence: {confidence:.2f}%\n"
                f"Processing Time: {processing_time:.2f} seconds\n\n"
                f"Findings:\n{result_detail}\n\n"
                f"Recommendation: {recommendation}"
            )
            
        except Exception as e:
            messagebox.showerror("Error", f"Prediction failed: {str(e)}")
            status_label.config(text="Prediction failed", fg="red")
            info_label.config(text="Error occurred during prediction", fg="red")

def clear_results():
    """Clear all results and reset GUI"""
    global last_image_path, last_prediction, confidence_score
    
    last_image_path = None
    last_prediction = None
    confidence_score = None
    
    # Clear image display
    for widget in canvas_frame.winfo_children():
        widget.destroy()
    
    # Add placeholder back
    placeholder_label = tk.Label(
        canvas_frame, 
        text="🖼️\nNo Image Loaded\n\nSelect an image to classify",
        font=("Helvetica", 12),
        fg="#bdc3c7",
        bg="white",
        justify=tk.CENTER
    )
    placeholder_label.pack(expand=True)
    
    # Reset labels
    result_label.config(text="No prediction yet", fg="#7f8c8d")
    detail_label.config(text="Load an image to classify", fg="#7f8c8d")
    info_label.config(text="Ready to classify bone marrow images", fg="#7f8c8d")
    status_label.config(text="Ready - Select an image to classify", fg="green")

# Load Models
def load_models():
    """Load ConvNeXtXLarge feature extractor and Logistic Regression classifier"""
    try:
        status_label.config(text="Loading ConvNeXtXLarge model from Keras...", fg="blue")
        root.update()
        
        # Load ConvNeXtXLarge directly from keras.applications (not from H5 file)
        # This avoids the custom layer issue
        feature_extractor = ConvNeXtXLarge(
            weights='imagenet', 
            include_top=False, 
            input_shape=(224, 224, 3)
        )
        
        # Freeze the model to prevent training
        feature_extractor.trainable = False
        
        status_label.config(text="Loading Logistic Regression classifier...", fg="blue")
        root.update()
        
        # Load Logistic Regression model
        lr_model = joblib.load('logistic_regression_bone_classifier.pkl')
        
        status_label.config(text="Models loaded successfully!", fg="green")
        return feature_extractor, lr_model
        
    except Exception as e:
        messagebox.showerror(
            "Error", 
            f"Failed to load models:\n{str(e)}\n\n"
            "Make sure the following file is in the same directory:\n"
            "1. 'logistic_regression_bone_classifier.pkl'\n\n"
            "Note: ConvNeXtXLarge will be loaded directly from Keras with ImageNet weights."
        )
        status_label.config(text="Model loading failed", fg="red")
        return None, None

# Create main GUI window
root = tk.Tk()
root.title("Bone Marrow Changes Lumbar Vertebrae Classifier System")
root.geometry("900x750")
root.config(bg="#f5f6fa")
root.resizable(True, True)

# Set custom style
style = {
    'bg_color': '#f5f6fa',
    'primary_color': '#3498db',
    'secondary_color': '#2ecc71',
    'danger_color': '#e74c3c',
    'text_color': '#2c3e50',
    'border_color': '#bdc3c7'
}

# Title Frame
title_frame = tk.Frame(root, bg=style['bg_color'])
title_frame.pack(fill=tk.X, pady=15)

# Title
title = tk.Label(
    title_frame, 
    text="🦴 Bone Marrow Changes Lumbar Vertebrae Classifier System", 
    font=("Helvetica", 16, "bold"), 
    bg=style['bg_color'], 
    fg=style['text_color']
)
title.pack()

subtitle = tk.Label(
    title_frame, 
    text="Using ConvNeXtXLarge + Logistic Regression Classifier", 
    font=("Helvetica", 10), 
    bg=style['bg_color'], 
    fg="#7f8c8d"
)
subtitle.pack(pady=(5, 0))

# Separator
separator = tk.Frame(root, height=2, bg=style['border_color'])
separator.pack(fill=tk.X, padx=20, pady=10)

# Result Labels Frame
result_frame = tk.Frame(root, bg=style['bg_color'])
result_frame.pack(pady=10)

result_label = tk.Label(
    result_frame, 
    text="No prediction yet", 
    font=("Helvetica", 18, "bold"),
    bg=style['bg_color'], 
    fg="#7f8c8d"
)
result_label.pack()

detail_label = tk.Label(
    result_frame, 
    text="Load an image to classify", 
    font=("Helvetica", 11),
    bg=style['bg_color'], 
    fg="#7f8c8d"
)
detail_label.pack(pady=5)

info_label = tk.Label(
    result_frame, 
    text="Ready to classify bone marrow images", 
    font=("Helvetica", 10, "italic"),
    bg=style['bg_color'], 
    fg="#7f8c8d"
)
info_label.pack(pady=5)

# Buttons Frame
button_frame = tk.Frame(root, bg=style['bg_color'])
button_frame.pack(pady=15)

# Style for buttons
button_style = {
    'font': ("Helvetica", 11, "bold"),
    'fg': "white",
    'padx': 20,
    'pady': 10,
    'width': 18,
    'cursor': "hand2",
    'relief': tk.RAISED,
    'bd': 1
}

browse_btn = tk.Button(
    button_frame, 
    text="📁 Browse & Classify", 
    command=browse_and_predict,
    bg=style['primary_color'],
    **button_style
)
browse_btn.pack(side=tk.LEFT, padx=10)

clear_btn = tk.Button(
    button_frame, 
    text="🗑️ Clear Results", 
    command=clear_results,
    bg=style['danger_color'],
    **button_style
)
clear_btn.pack(side=tk.LEFT, padx=10)

# Image Display Frame with border
canvas_container = tk.Frame(root, bg=style['border_color'], relief=tk.RAISED, bd=2)
canvas_container.pack(pady=20, padx=20, ipadx=2, ipady=2)

canvas_frame = tk.Frame(canvas_container, bg="white", width=400, height=400)
canvas_frame.pack()
canvas_frame.pack_propagate(False)

# Add placeholder text when no image
placeholder_label = tk.Label(
    canvas_frame, 
    text="🖼️\nNo Image Loaded\n\nSelect an image to classify",
    font=("Helvetica", 12),
    fg="#bdc3c7",
    bg="white",
    justify=tk.CENTER
)
placeholder_label.pack(expand=True)

# Information Frame
info_frame = tk.Frame(root, bg=style['bg_color'])
info_frame.pack(pady=10, padx=20, fill=tk.X)

info_text = tk.Label(
    info_frame,
    text="ℹ️ Information:\n• Supported formats: JPG, PNG, JPEG, BMP, TIFF\n• Image will be resized to 224x224 pixels\n• Results include confidence score and processing time\n• Using ConvNeXtXLarge pre-trained on ImageNet",
    font=("Helvetica", 9),
    bg=style['bg_color'],
    fg="#7f8c8d",
    justify=tk.LEFT
)
info_text.pack()

# Status Bar
status_label = tk.Label(
    root, 
    text="Initializing...", 
    bd=1, 
    relief=tk.SUNKEN,
    anchor=tk.W, 
    bg="#ecf0f1", 
    font=("Helvetica", 9), 
    fg="blue"
)
status_label.pack(side=tk.BOTTOM, fill=tk.X)

# Load models
feature_extractor, lr_model = load_models()

# Start GUI
if feature_extractor is not None and lr_model is not None:
    status_label.config(text="Ready - Select an image to classify", fg="green")
    root.mainloop()
else:
    messagebox.showerror("Error", "Failed to load required models. Application will exit.")
    root.destroy()

C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
